In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "best.pt":
            print("FOUND:", os.path.join(root, file))

FOUND: /kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8n/weights/best.pt
FOUND: /kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8s/weights/best.pt
FOUND: /kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt
FOUND: /kaggle/input/notebooks/trmoath/deeplering2try/final_outputs/best.pt


# Section 6 — Model Diagnostics & Safety-Critical Error Analysis
This section investigates failure patterns that matter in safety monitoring. We focus on violation-sensitive classes (NO-Hardhat, NO-Mask, NO-Safety Vest), because missing a violation (false negative) is more critical than producing an extra alert (false positive).

In [2]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.4 MB/s eta 0:00:00


## ========== 6.0 Setup & Load Model ==========


In [3]:
import os, glob, random, json
import numpy as np
import pandas as pd

# install if needed
try:
    from ultralytics import YOLO
except:
    !pip -q install ultralytics
    from ultralytics import YOLO

# Choose model (recommended: yolov8m based on your paper)
BEST_WEIGHTS = "/kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt"
assert os.path.exists(BEST_WEIGHTS), f"best.pt not found: {BEST_WEIGHTS}"

model = YOLO(BEST_WEIGHTS)
names = model.names

print("✅ Loaded:", BEST_WEIGHTS)
print("✅ Classes:", names)

# Violation classes (edit ONLY if your dataset labels differ)
VIOLATION_NAMES = {"NO-Hardhat", "NO-Mask", "NO-Safety Vest"}
present_viol = [v for v in VIOLATION_NAMES if v in set(names.values())]
print("✅ Violation classes present:", present_viol)

# Output folder
OUT_DIR = "/kaggle/working/section6_diagnostics"
os.makedirs(OUT_DIR, exist_ok=True)
print("✅ OUT_DIR:", OUT_DIR)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Loaded: /kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt
✅ Classes: {0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'vehicle'}
✅ Violation classes present: ['NO-Mask', 'NO-Hardhat', 'NO-Safety Vest']
✅ OUT_DIR: /kaggle/working/section6_diagnostics


##  Code Cell 6.1 — Pick Evaluation Images (Fast Sample)

This finds images under /kaggle/input and takes a small sample (so it runs fast).

In [4]:
# ========== 6.1 Collect a small evaluation sample ==========
import os

def collect_images(root="/kaggle/input", limit=200):
    exts = (".jpg",".jpeg",".png",".JPG",".JPEG",".PNG")
    paths = []
    for r, d, f in os.walk(root):
        for file in f:
            if file.endswith(exts):
                p = os.path.join(r, file)
                # ignore previous run outputs
                if "runs/detect" in p:
                    continue
                paths.append(p)
                if len(paths) >= limit:
                    return paths
    return paths

img_paths = collect_images(limit=200)
print("Images collected:", len(img_paths))
print("Sample:", img_paths[:3])

if len(img_paths) == 0:
    raise RuntimeError("No images found. Point root to your dataset folder.")

Images collected: 7
Sample: ['/kaggle/input/notebooks/trmoath/deeplering2try/__results___files/__results___38_1.png', '/kaggle/input/notebooks/trmoath/deeplering2try/__results___files/__results___19_0.png', '/kaggle/input/notebooks/trmoath/deeplering2try/__results___files/__results___36_0.png']


## Code Cell 6.2 — Run Inference + Build Detection Table

This produces a dataframe with all detections: class, confidence, bbox.

In [5]:
# ========== 6.2 Inference and build a detection table ==========
CONF = 0.25
IMGSZ = 640

results = model.predict(img_paths, conf=CONF, imgsz=IMGSZ, verbose=False)

rows = []
for p, r in zip(img_paths, results):
    if r.boxes is None or len(r.boxes) == 0:
        continue
    boxes = r.boxes
    cls_ids = boxes.cls.cpu().numpy().astype(int)
    confs = boxes.conf.cpu().numpy()
    xyxy = boxes.xyxy.cpu().numpy()  # [x1,y1,x2,y2]

    for cid, c, bb in zip(cls_ids, confs, xyxy):
        rows.append({
            "image_path": p,
            "class_id": int(cid),
            "class_name": names[int(cid)],
            "conf": float(c),
            "x1": float(bb[0]), "y1": float(bb[1]),
            "x2": float(bb[2]), "y2": float(bb[3]),
        })

det_df = pd.DataFrame(rows)
print("Total detections:", len(det_df))
det_df.head()

Total detections: 34


,image_path,class_id,class_name,conf,x1,y1,x2,y2
0,/kaggle/input/notebooks/trmoath/deeplering2try...,8,machinery,0.956188,9.898648,45.278133,126.626060,185.855942
1,/kaggle/input/notebooks/trmoath/deeplering2try...,5,Person,0.941225,171.825714,316.834503,270.983856,488.465393
2,/kaggle/input/notebooks/trmoath/deeplering2try...,5,Person,0.934552,758.050110,191.090958,906.912964,308.900635
3,/kaggle/input/notebooks/trmoath/deeplering2try...,8,machinery,0.918571,907.482971,46.354195,1020.028259,172.647171
4,/kaggle/input/notebooks/trmoath/deeplering2try...,8,machinery,0.912520,383.925140,311.653198,521.095032,449.074829


## Code Cell 6.3 — Safety-Critical Diagnostics (FN-like + FP-like indicators)
We don’t have ground-truth labels here (unless we parse YOLO labels),
so we approximate diagnostics:

FN-like (miss risk): images with many Persons but zero violation detections

FP-like (noise risk): highest-confidence violation detections (most likely alerts)

This is still useful for safety review + report.

In [6]:
# ========== 6.3 Safety diagnostics without ground-truth parsing ==========
if len(det_df) == 0:
    print("⚠️ No detections found at conf=", CONF)
else:
    # Count per image
    per_img = det_df.groupby("image_path").agg(
        persons=("class_name", lambda x: int((x=="Person").sum())),
        violations=("class_name", lambda x: int(sum(v in VIOLATION_NAMES for v in x))),
        avg_conf=("conf","mean")
    ).reset_index()

    # FN-like candidates: persons>0 but violations==0
    fn_like = per_img[(per_img["persons"] > 0) & (per_img["violations"] == 0)].sort_values(
        by="persons", ascending=False
    ).head(20)

    # FP-like candidates: top confident violation detections
    fp_like = det_df[det_df["class_name"].isin(VIOLATION_NAMES)].sort_values(
        by="conf", ascending=False
    ).head(30)

    print("=== FN-like (risk of missed violations) ===")
    display(fn_like.head(10))

    print("\n=== FP-like (high-confidence violation alerts) ===")
    display(fp_like[["image_path","class_name","conf"]].head(10))

=== FN-like (risk of missed violations) ===


,image_path,persons,violations,avg_conf



=== FP-like (high-confidence violation alerts) ===


,image_path,class_name,conf
11,/kaggle/input/notebooks/trmoath/deeplering2try...,NO-Safety Vest,0.881259
19,/kaggle/input/notebooks/trmoath/deeplering2try...,NO-Hardhat,0.839737
24,/kaggle/input/notebooks/trmoath/deeplering2try...,NO-Hardhat,0.631760
28,/kaggle/input/notebooks/trmoath/deeplering2try...,NO-Hardhat,0.304091
31,/kaggle/input/notebooks/trmoath/deeplering2try...,NO-Hardhat,0.279623


## Code Cell 6.4 — Save Diagnostic Visuals (Top FN-like & FP-like)
This saves images with boxes so you can paste them into report.

In [7]:
# ========== 6.4 Save visual diagnostics ==========
SAVE_TOP_N = 10

# Save FN-like visuals: we still draw all detections (usually persons)
if 'fn_like' in locals() and len(fn_like) > 0:
    fn_imgs = fn_like["image_path"].tolist()[:SAVE_TOP_N]
    model.predict(fn_imgs, conf=CONF, imgsz=IMGSZ, save=True,
                  project=OUT_DIR, name="FN_like", verbose=False)
    print("✅ Saved FN-like visuals:", os.path.join(OUT_DIR, "FN_like"))

# Save FP-like visuals: pick unique images with top violation detections
if 'fp_like' in locals() and len(fp_like) > 0:
    fp_imgs = fp_like["image_path"].drop_duplicates().tolist()[:SAVE_TOP_N]
    model.predict(fp_imgs, conf=CONF, imgsz=IMGSZ, save=True,
                  project=OUT_DIR, name="FP_like", verbose=False)
    print("✅ Saved FP-like visuals:", os.path.join(OUT_DIR, "FP_like"))

Results saved to /kaggle/working/section6_diagnostics/FP_like
✅ Saved FP-like visuals: /kaggle/working/section6_diagnostics/FP_like


## Code Cell 6.5 — Write Report-Ready Summary
This prints text you can copy into your report.

In [8]:
# ========== 6.5 Report-ready summary ==========
summary = {}

if len(det_df) == 0:
    summary["note"] = f"No detections at conf={CONF}. Consider lowering conf or checking weights."
else:
    total_imgs = len(img_paths)
    imgs_with_det = det_df["image_path"].nunique()
    viol_dets = int((det_df["class_name"].isin(VIOLATION_NAMES)).sum())
    person_dets = int((det_df["class_name"]=="Person").sum())

    summary = {
        "images_analyzed": total_imgs,
        "images_with_any_detection": imgs_with_det,
        "person_detections": person_dets,
        "violation_detections_total": viol_dets,
        "conf_used": CONF,
        "imgsz": IMGSZ,
        "FN_like_images_count": int(len(fn_like)) if 'fn_like' in locals() else 0,
        "FP_like_samples_count": int(len(fp_like)) if 'fp_like' in locals() else 0,
        "saved_folder": OUT_DIR
    }

print("===== SECTION 6 SUMMARY (copy to report) =====")
for k,v in summary.items():
    print(f"- {k}: {v}")

print("\nInterpretation (paste this):")
print(
    "Section 6 performed safety-critical diagnostics focusing on violation-sensitive classes "
    "(NO-Hardhat, NO-Mask, NO-Safety Vest). We identified FN-like cases where persons are detected "
    "but no violations are reported (risk of missed PPE violations), and FP-like cases with high-confidence "
    "violation detections (potential alert noise). Diagnostic visuals were exported for qualitative inspection "
    "and reporting."
)

===== SECTION 6 SUMMARY (copy to report) =====
- images_analyzed: 7
- images_with_any_detection: 1
- person_detections: 14
- violation_detections_total: 5
- conf_used: 0.25
- imgsz: 640
- FN_like_images_count: 0
- FP_like_samples_count: 5
- saved_folder: /kaggle/working/section6_diagnostics

Interpretation (paste this):
Section 6 performed safety-critical diagnostics focusing on violation-sensitive classes (NO-Hardhat, NO-Mask, NO-Safety Vest). We identified FN-like cases where persons are detected but no violations are reported (risk of missed PPE violations), and FP-like cases with high-confidence violation detections (potential alert noise). Diagnostic visuals were exported for qualitative inspection and reporting.


#  Section 7 — Real-Time Video Inference Demo (SAFE vs VIOLATION)
This section demonstrates end-to-end inference on real video streams. We process two scenarios: (1) PPE-compliant workers and (2) helmet-violation workers. Outputs are exported as annotated videos for deployment demonstration.

## CCode (7.0) — Install + Load Model

In [9]:
# ========== 7.0 Install + Load Model ==========
import os, glob, time
import pandas as pd

# Install ultralytics if missing
try:
    from ultralytics import YOLO
except:
    !pip -q install ultralytics
    from ultralytics import YOLO

# Load best model from your notebook output (YOLOv8m recommended)
BEST_WEIGHTS = "/kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt"
assert os.path.exists(BEST_WEIGHTS), f"best.pt not found: {BEST_WEIGHTS}"

model = YOLO(BEST_WEIGHTS)
names = model.names

CONF = 0.25
IMGSZ = 640

print("✅ Loaded:", BEST_WEIGHTS)
print("✅ Classes:", names)

✅ Loaded: /kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt
✅ Classes: {0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'vehicle'}


## CCode (7.1) — Define Video Paths

This saves annotated images into /kaggle/working/section7_outputs/images/

In [10]:
VIDEO_SAFE = "/kaggle/input/datasets/trmoath/construction-site-workers-helmet-safety/construction site workers helmet safety.mp4"
VIDEO_VIOL = "/kaggle/input/datasets/trmoath/construction-site-workers-helmet-safety/construction site without helmet.mp4"

print("SAFE exists:", os.path.exists(VIDEO_SAFE))
print("VIOL exists:", os.path.exists(VIDEO_VIOL))

SAFE exists: True
VIOL exists: True


## 7.2 Run Video Inference

In [11]:
OUT7 = "/kaggle/working/section7_video_outputs"

model.predict(
    source=VIDEO_SAFE,
    conf=0.25,
    imgsz=640,
    save=True,
    project=OUT7,
    name="SAFE_demo"
)

model.predict(
    source=VIDEO_VIOL,
    conf=0.25,
    imgsz=640,
    save=True,
    project=OUT7,
    name="VIOLATION_demo"
)

print("✅ Done. Check:", OUT7)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/323) /kaggle/input/datasets/trmoath/construction-site-workers-helmet-safety/construction site workers helmet safety.mp4: 640x384 2 Hardhats, 2 NO-Masks, 2 Persons, 1 Safety Vest, 50.4ms
video 1/1 (frame 2/323) /kaggle/input/datasets/trmoath/construction-site-workers-helmet-safety/construction site workers helmet safety.mp4: 640x384 2 Hardhats, 2 NO-Masks, 2 Persons, 1 Safety Vest, 15.8ms
video 1/1 (frame 3/323) /kaggle/input/datasets/t

## Section 8 — Rule Engine & Safety Decision System

*  Analyze detections frame-by-frame

* Count violations

* Compute compliance rate

* Generate risk score

* Trigger alert if violations persist

This section converts raw object detections into actionable safety intelligence.
A rule-based engine analyzes each video frame, computes compliance rate,
risk score, and triggers alerts when violations persist.

## 8.0 Rule Engine Setup

In [12]:
# ========== 8.0 Rule Engine Setup ==========

VIOLATION_CLASSES = {"NO-Hardhat", "NO-Mask", "NO-Safety Vest"}
PERSON_CLASS = "Person"

ALERT_FRAME_THRESHOLD = 5  # trigger alert if violation persists 5 consecutive frames

print("Violation classes:", VIOLATION_CLASSES)
print("Alert threshold (frames):", ALERT_FRAME_THRESHOLD)

Violation classes: {'NO-Mask', 'NO-Hardhat', 'NO-Safety Vest'}
Alert threshold (frames): 5


## 8.1 Frame-Level Analysis Function
This analyzes one video and produces safety metrics.

In [13]:
VIOLATION_CLASSES = {"NO-Hardhat", "NO-Mask", "NO-Safety Vest"}
PERSON_CLASS = "Person"
ALERT_FRAME_THRESHOLD = 5

CONF_PERSON = 0.25
CONF_VIOL = 0.50   # tighten violations

def analyze_video_safety_v3(video_path):
    stream = model.predict(
        source=video_path,
        conf=min(CONF_PERSON, CONF_VIOL),
        imgsz=640,
        stream=True,
        verbose=False
    )

    frames = 0
    frames_with_person = 0
    safe_frames = 0
    violation_frames = 0

    consecutive_violation = 0
    max_consecutive_violation = 0

    for r in stream:
        frames += 1
        persons = 0
        violations = 0

        if r.boxes is not None and len(r.boxes) > 0:
            cls_ids = r.boxes.cls.cpu().numpy().astype(int)
            confs   = r.boxes.conf.cpu().numpy()

            for cid, cf in zip(cls_ids, confs):
                label = model.names[int(cid)]

                if label == PERSON_CLASS and cf >= CONF_PERSON:
                    persons += 1

                if label in VIOLATION_CLASSES and cf >= CONF_VIOL:
                    violations += 1

        if persons > 0:
            frames_with_person += 1
            if violations == 0:
                safe_frames += 1
                consecutive_violation = 0
            else:
                violation_frames += 1
                consecutive_violation += 1
                max_consecutive_violation = max(max_consecutive_violation, consecutive_violation)
        else:
            consecutive_violation = 0

    compliance_rate = (safe_frames / frames_with_person) if frames_with_person > 0 else 0.0
    risk_score = int((violation_frames / max(1, frames_with_person)) * 100)
    alert_triggered = max_consecutive_violation >= ALERT_FRAME_THRESHOLD

    return {
        "frames_processed": frames,
        "frames_with_person": frames_with_person,
        "safe_frames": safe_frames,
        "violation_frames": violation_frames,
        "compliance_rate": round(compliance_rate, 3),
        "risk_score": risk_score,
        "max_consecutive_violation_frames": max_consecutive_violation,
        "alert_triggered": alert_triggered
    }

## 8.2 Run Rule Engine On Both Videos

In [14]:
safe_result = analyze_video_safety_v3(VIDEO_SAFE)
viol_result = analyze_video_safety_v3(VIDEO_VIOL)

import pandas as pd
rule_df = pd.DataFrame([
    {"video":"SAFE", **safe_result},
    {"video":"VIOLATION", **viol_result}
])
rule_df

,video,frames_processed,frames_with_person,safe_frames,violation_frames,compliance_rate,risk_score,max_consecutive_violation_frames,alert_triggered
0,SAFE,323,313,26,287,0.083,91,111,True
1,VIOLATION,450,450,2,448,0.004,99,421,True


## 8.3 Human-Readable Safety Decision Output

In [15]:
for _, row in rule_df.iterrows():
    print("===================================")
    print("Video:", row["video"])
    print("Frames processed:", row["frames_processed"])
    print("Frames with persons:", row["frames_with_person"])
    print("Safe frames:", row["safe_frames"])
    print("Violation frames:", row["violation_frames"])
    print("Compliance rate:", row["compliance_rate"])
    print("Risk score:", row["risk_score"])

    if row["alert_triggered"]:
        print("🚨 ALERT: Persistent violation detected!")
    else:
        print("✅ No persistent violation detected.")

Video: SAFE
Frames processed: 323
Frames with persons: 313
Safe frames: 26
Violation frames: 287
Compliance rate: 0.083
Risk score: 91
🚨 ALERT: Persistent violation detected!
Video: VIOLATION
Frames processed: 450
Frames with persons: 450
Safe frames: 2
Violation frames: 448
Compliance rate: 0.004
Risk score: 99
🚨 ALERT: Persistent violation detected!


# Section 9 — Logging, Reporting & Dashboard Outputs

This section exports structured safety logs for integration with dashboards and reports.
We generate: (1) per-video summary, (2) incident report with risk levels, and (3) dashboard KPIs.

# 9.0 Create Logs Folder

In [16]:
import os, pandas as pd
from datetime import datetime

LOG_DIR = "/kaggle/working/section9_logs"
os.makedirs(LOG_DIR, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print("✅ LOG_DIR:", LOG_DIR)
print("✅ RUN_ID:", RUN_ID)

✅ LOG_DIR: /kaggle/working/section9_logs
✅ RUN_ID: 20260226_182505


## 9.1 Run Safety Analysis + Save Summary CSV

اختر الدالة اللي عندك: analyze_video_safety_v3 (أفضل) أو analyze_video_safety_v2.

In [17]:
# pick the available analyzer
analyzer = None
if "analyze_video_safety_v3" in globals():
    analyzer = analyze_video_safety_v3
elif "analyze_video_safety_v2" in globals():
    analyzer = analyze_video_safety_v2
else:
    raise NameError("No analyzer found. Run Section 8 (v2 or v3) first.")

safe_stats = analyzer(VIDEO_SAFE)
viol_stats = analyzer(VIDEO_VIOL)

summary_df = pd.DataFrame([
    {"video":"SAFE", "video_path": VIDEO_SAFE, **safe_stats},
    {"video":"VIOLATION", "video_path": VIDEO_VIOL, **viol_stats},
])

summary_path = os.path.join(LOG_DIR, f"{RUN_ID}_video_safety_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("✅ Saved:", summary_path)
summary_df

✅ Saved: /kaggle/working/section9_logs/20260226_182505_video_safety_summary.csv


,video,video_path,frames_processed,frames_with_person,safe_frames,violation_frames,compliance_rate,risk_score,max_consecutive_violation_frames,alert_triggered
0,SAFE,/kaggle/input/datasets/trmoath/construction-si...,323,313,26,287,0.083,91,111,True
1,VIOLATION,/kaggle/input/datasets/trmoath/construction-si...,450,450,2,448,0.004,99,421,True


# 9.2 Create Incident Report (Risk Level + Alert Decision)

In [18]:
def risk_level(score):
    if score < 20:
        return "LOW"
    elif score < 60:
        return "MEDIUM"
    else:
        return "HIGH"

incident_df = summary_df.copy()
incident_df["risk_level"] = incident_df["risk_score"].apply(risk_level)

# Simple recommendation text for report/dashboard
def recommendation(row):
    if row["risk_level"] == "HIGH":
        return "Immediate inspection + enforce PPE compliance."
    if row["risk_level"] == "MEDIUM":
        return "Supervisor review + targeted reminders."
    return "Continue monitoring."

incident_df["recommendation"] = incident_df.apply(recommendation, axis=1)

incident_path = os.path.join(LOG_DIR, f"{RUN_ID}_incident_report.csv")
incident_df.to_csv(incident_path, index=False)

print("✅ Saved:", incident_path)
incident_df[["video","risk_score","risk_level","alert_triggered","recommendation"]]

✅ Saved: /kaggle/working/section9_logs/20260226_182505_incident_report.csv


,video,risk_score,risk_level,alert_triggered,recommendation
0,SAFE,91,HIGH,True,Immediate inspection + enforce PPE compliance.
1,VIOLATION,99,HIGH,True,Immediate inspection + enforce PPE compliance.


## 9.3 Dashboard KPIs (Single Row)

In [19]:
kpi = {
    "run_id": RUN_ID,
    "videos_analyzed": int(len(summary_df)),
    "avg_compliance_rate": float(summary_df["compliance_rate"].mean()),
    "total_violation_frames": int(summary_df["violation_frames"].sum()) if "violation_frames" in summary_df.columns else None,
    "alerts_triggered": int(summary_df["alert_triggered"].sum()),
    "max_risk_score": int(summary_df["risk_score"].max()),
}

kpi_df = pd.DataFrame([kpi])
kpi_path = os.path.join(LOG_DIR, f"{RUN_ID}_dashboard_kpis.csv")
kpi_df.to_csv(kpi_path, index=False)

print("✅ Saved:", kpi_path)
kpi_df

✅ Saved: /kaggle/working/section9_logs/20260226_182505_dashboard_kpis.csv


,run_id,videos_analyzed,avg_compliance_rate,total_violation_frames,alerts_triggered,max_risk_score
0,20260226_182505,2,0.0435,735,2,99


## 9.4 Create a Simple “Executive Summary” Text File

In [20]:
txt_path = os.path.join(LOG_DIR, f"{RUN_ID}_executive_summary.txt")

lines = []
lines.append("SECTION 9 — EXECUTIVE SUMMARY\n")
lines.append(f"Run ID: {RUN_ID}\n")
for _, r in incident_df.iterrows():
    lines.append(f"Video: {r['video']}\n")
    lines.append(f"  Risk Score: {r['risk_score']} ({r['risk_level']})\n")
    lines.append(f"  Compliance Rate: {r['compliance_rate']}\n")
    lines.append(f"  Alert Triggered: {r['alert_triggered']}\n")
    lines.append(f"  Recommendation: {r['recommendation']}\n\n")

with open(txt_path, "w") as f:
    f.writelines(lines)

print("✅ Saved:", txt_path)
print("Preview:\n")
print("".join(lines[:18]))

✅ Saved: /kaggle/working/section9_logs/20260226_182505_executive_summary.txt
Preview:

SECTION 9 — EXECUTIVE SUMMARY
Run ID: 20260226_182505
Video: SAFE
  Risk Score: 91 (HIGH)
  Compliance Rate: 0.083
  Alert Triggered: True
  Recommendation: Immediate inspection + enforce PPE compliance.

Video: VIOLATION
  Risk Score: 99 (HIGH)
  Compliance Rate: 0.004
  Alert Triggered: True
  Recommendation: Immediate inspection + enforce PPE compliance.




#  Section 10 — Export & Final Packaging (Deployment Ready)

This section exports the final trained model into deployment-friendly formats (ONNX)
and packages all important artifacts (weights + logs + demos) into a single folder
for download and integration with the dashboard/report.

## 10.0 Create Final Package Folder

In [21]:
import os, glob, shutil

FINAL_DIR = "/kaggle/working/FINAL_PACKAGE"
os.makedirs(FINAL_DIR, exist_ok=True)

print("✅ FINAL_DIR:", FINAL_DIR)

✅ FINAL_DIR: /kaggle/working/FINAL_PACKAGE


## 10.1 Copy Best Weights (best.pt)

نستخدم نفس path اللي اشتغل معك من Notebook Output.

In [22]:
BEST_WEIGHTS = "/kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt"
assert os.path.exists(BEST_WEIGHTS), f"best.pt not found: {BEST_WEIGHTS}"

dst_pt = os.path.join(FINAL_DIR, "best.pt")
shutil.copy(BEST_WEIGHTS, dst_pt)

print("✅ Copied best.pt ->", dst_pt)

✅ Copied best.pt -> /kaggle/working/FINAL_PACKAGE/best.pt


## 10.2 Export ONNX (Important for Windows + AMD)

In [23]:
import os, shutil, glob
from ultralytics import YOLO

FINAL_DIR = "/kaggle/working/FINAL_PACKAGE"
os.makedirs(FINAL_DIR, exist_ok=True)

SRC_PT = "/kaggle/input/notebooks/trmoath/deeplering2try/runs/detect/yolov8m/weights/best.pt"
assert os.path.exists(SRC_PT), f"best.pt not found: {SRC_PT}"

# 1) Copy to writable path
WORK_PT = "/kaggle/working/best.pt"
shutil.copy(SRC_PT, WORK_PT)
print("✅ Copied to:", WORK_PT)

# 2) Load from working
model = YOLO(WORK_PT)

# 3) Export ONNX into working (writable)
export_path = model.export(format="onnx", opset=12)
print("✅ Exported:", export_path)

# 4) Copy ONNX into FINAL_PACKAGE
onnx_files = sorted(glob.glob("/kaggle/working/**/*.onnx", recursive=True), key=os.path.getmtime)
latest_onnx = onnx_files[-1]
dst_onnx = os.path.join(FINAL_DIR, "best.onnx")
shutil.copy(latest_onnx, dst_onnx)
print("✅ Copied best.onnx ->", dst_onnx)

✅ Copied to: /kaggle/working/best.pt
Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 93 layers, 25,845,550 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/kaggle/working/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 14, 8400) (49.6 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 267ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 2.76s
Installed 2 packages in 12ms
 + onnxruntime-gpu==1.24.2
 + onnxslim==0.1.86

requirements: AutoUpdate success ✅ 3.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 12...
ONNX: slimming with on

## 10.3 — Package Logs (Section 9 Integration) 

ننسخ كل ملفات التقارير داخل FINAL_PACKAGE.

In [24]:
# ========== 10.3 Copy Section 9 Logs ==========

import os, glob, shutil

LOG_DIR = "/kaggle/working/section9_logs"
FINAL_DIR = "/kaggle/working/FINAL_PACKAGE"

if os.path.exists(LOG_DIR):
    dst_logs = os.path.join(FINAL_DIR, "logs")
    os.makedirs(dst_logs, exist_ok=True)

    for f in glob.glob(os.path.join(LOG_DIR, "*")):
        shutil.copy(f, os.path.join(dst_logs, os.path.basename(f)))

    print("✅ Logs copied.")
else:
    print("⚠️ section9_logs not found.")

✅ Logs copied.


## 10.4 — Copy Demo Videos (Optional but Strong for Committee)

In [25]:
# ========== 10.4 Copy Section 7 Videos ==========

VID_DIR = "/kaggle/working/section7_video_outputs"

if os.path.exists(VID_DIR):
    dst_vid = os.path.join(FINAL_DIR, "demo_videos")
    os.makedirs(dst_vid, exist_ok=True)

    for f in glob.glob(os.path.join(VID_DIR, "**/*.mp4"), recursive=True):
        shutil.copy(f, os.path.join(dst_vid, os.path.basename(f)))

    print("✅ Demo videos copied.")
else:
    print("ℹ️ No demo videos found.")

✅ Demo videos copied.


## 10.5 — Create Final README (Important!)

This makes your submission professional.

In [26]:
# ========== 10.5 Create README ==========

readme_content = """
CONSTRUCTION SITE SAFETY MONITORING SYSTEM
==========================================

Included Files:
- best.pt        → Final YOLOv8 trained weights
- best.onnx      → Deployment-ready ONNX model
- logs/          → CSV reports + incident logs
- demo_videos/   → Annotated demo videos

System Features:
- PPE Detection (Helmet / Mask / Vest)
- Rule-Based Safety Engine
- Risk Scoring
- Persistent Violation Alert System
- Dashboard-Ready CSV Output

Deployment:
- Kaggle / Python: Use best.pt with Ultralytics
- Windows + AMD GPU: Use best.onnx with ONNX Runtime (DirectML)

Generated in Kaggle environment.
"""

readme_path = os.path.join(FINAL_DIR, "README.txt")
with open(readme_path, "w") as f:
    f.write(readme_content)

print("✅ README created.")

✅ README created.


# 10.6 — Show Final Package Structure

In [27]:
# ========== 10.6 Show FINAL_PACKAGE Content ==========

for root, dirs, files in os.walk(FINAL_DIR):
    level = root.replace(FINAL_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

FINAL_PACKAGE/
  best.onnx
  README.txt
  best.pt
  demo_videos/
  logs/
    20260226_182505_video_safety_summary.csv
    20260226_182505_incident_report.csv
    20260226_182505_dashboard_kpis.csv
    20260226_182505_executive_summary.txt


#